## Importing libraries


In [ ]:
import os

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn

from google.colab import drive


SEED = 42

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

WORK_START = 8
WORK_END = 18

TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15

STREAM_THRESHOLD_PERCENTILE = 0.95
ENSEMBLE_THRESHOLD_PERCENTILE = 0.95

OCEAN_LOW_PERCENTILE = 0.05
OCEAN_HIGH_PERCENTILE = 0.95

EPOCHS = 50
PATIENCE = 7
LEARNING_RATE = 0.001
BATCH_SIZE = 2048

DATE_FORMAT = "%m/%d/%Y %H:%M:%S"

print("Training device:", DEVICE)


Training device: cpu


## Mount Google Drive and locate the CERT files


In [ ]:
drive.mount("/content/drive")

DRIVE_ROOT = "/content/drive/MyDrive"

required_names = {
    "logon.csv",
    "device.csv",
    "file.csv",
    "psychometric.csv"
}

matching_folders = []

for current_folder, subfolders, filenames in os.walk(DRIVE_ROOT):
    lowercase_names = {
        filename.lower()
        for filename in filenames
    }

    if required_names.issubset(lowercase_names):
        matching_folders.append(current_folder)

if len(matching_folders) == 0:
    raise FileNotFoundError(
        "No My Drive folder contains logon.csv, device.csv, "
        "file.csv and psychometric.csv together. If the folder "
        "is shared, add a shortcut to My Drive and run again."
    )

if len(matching_folders) > 1:
    print("More than one matching folder was found:")

    for folder in matching_folders:
        print(folder)

    raise ValueError(
        "Set DATA_FOLDER manually to the correct folder shown above."
    )

DATA_FOLDER = matching_folders[0]
OUTPUT_FOLDER = os.path.join(
    DATA_FOLDER,
    "CERT_model_outputs"
)

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)


def find_filename(folder, expected_name):
    for filename in os.listdir(folder):
        if filename.lower() == expected_name.lower():
            return os.path.join(folder, filename)

    raise FileNotFoundError(expected_name)


LOGON_PATH = find_filename(DATA_FOLDER, "logon.csv")
DEVICE_PATH = find_filename(DATA_FOLDER, "device.csv")
FILE_PATH = find_filename(DATA_FOLDER, "file.csv")
PSYCHOMETRIC_PATH = find_filename(
    DATA_FOLDER,
    "psychometric.csv"
)

required_files = {
    "logon.csv": LOGON_PATH,
    "device.csv": DEVICE_PATH,
    "file.csv": FILE_PATH,
    "psychometric.csv": PSYCHOMETRIC_PATH
}

print("Data folder:", DATA_FOLDER)
print("Output folder:", OUTPUT_FOLDER)

for filename, path in required_files.items():
    size_in_mb = os.path.getsize(path) / (1024 ** 2)

    print(
        filename,
        "-",
        round(size_in_mb, 2),
        "MB"
    )


Mounted at /content/drive
Data folder: /content/drive/MyDrive/r6.2
Output folder: /content/drive/MyDrive/r6.2/CERT_model_outputs
logon.csv - 230.45 MB
device.csv - 133.08 MB
file.csv - 1269.62 MB
psychometric.csv - 0.17 MB


## Load Selected CSV Columns

In [ ]:
def load_columns(path, required_columns):
    header = pd.read_csv(path, nrows=0)

    column_lookup = {
        column.strip().lower(): column
        for column in header.columns
    }

    missing_columns = [
        column
        for column in required_columns
        if column not in column_lookup
    ]

    if missing_columns:
        raise ValueError(
            f"{os.path.basename(path)} is missing: {missing_columns}. "
            f"Available columns: {list(header.columns)}"
        )

    original_column_names = [
        column_lookup[column]
        for column in required_columns
    ]

    dataframe = pd.read_csv(
        path,
        usecols=original_column_names,
        low_memory=False
    )

    dataframe.columns = [
        column.strip().lower()
        for column in dataframe.columns
    ]

    return dataframe

## Prepare timestamps and common event fields

In [ ]:
def prepare_events(dataframe):
    dataframe = dataframe.copy()

    parsed_dates = pd.to_datetime(
        dataframe["date"],
        format=DATE_FORMAT,
        errors="coerce"
    )

    if parsed_dates.isna().mean() > 0.01:
        parsed_dates = pd.to_datetime(
            dataframe["date"],
            errors="coerce"
        )

    dataframe["date"] = parsed_dates

    dataframe = dataframe.dropna(
        subset=["date", "user", "pc"]
    ).copy()

    dataframe["user"] = (
        dataframe["user"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    dataframe["pc"] = (
        dataframe["pc"]
        .astype(str)
        .str.strip()
    )

    dataframe["day"] = dataframe["date"].dt.normalize()
    dataframe["hour"] = dataframe["date"].dt.hour
    dataframe["weekday"] = dataframe["date"].dt.weekday

    dataframe["weekend"] = (
        dataframe["weekday"] >= 5
    ).astype(int)

    dataframe["after_hours"] = (
        (dataframe["hour"] < WORK_START)
        | (dataframe["hour"] >= WORK_END)
    ).astype(int)

    return dataframe

## Create logon features

In [ ]:
def build_logon_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["activity"] = (
        dataframe["activity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dataframe["is_logon"] = (
        dataframe["activity"] == "logon"
    ).astype(int)

    dataframe["is_logoff"] = (
        dataframe["activity"] == "logoff"
    ).astype(int)

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        logon_total=("activity", "size"),
        logon_count=("is_logon", "sum"),
        logoff_count=("is_logoff", "sum"),
        logon_unique_pcs=("pc", "nunique"),
        logon_first_hour=("hour", "min"),
        logon_last_hour=("hour", "max"),
        logon_after_hours=("after_hours", "sum"),
        logon_weekend=("weekend", "max"),
        logon_hour_std=("hour", "std")
    )

    daily["logon_hour_std"] = (
        daily["logon_hour_std"].fillna(0)
    )

    daily["logon_after_hours_ratio"] = (
        daily["logon_after_hours"]
        / daily["logon_total"]
    )

    daily["logon_logoff_ratio"] = (
        daily["logon_count"]
        / (daily["logoff_count"] + 1)
    )

    return daily

## Creating device features

In [ ]:
def build_device_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["activity"] = (
        dataframe["activity"]
        .astype(str)
        .str.strip()
        .str.lower()
    )

    dataframe["is_connect"] = (
        dataframe["activity"] == "connect"
    ).astype(int)

    dataframe["is_disconnect"] = (
        dataframe["activity"] == "disconnect"
    ).astype(int)

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        device_total=("activity", "size"),
        connect_count=("is_connect", "sum"),
        disconnect_count=("is_disconnect", "sum"),
        device_unique_pcs=("pc", "nunique"),
        device_first_hour=("hour", "min"),
        device_last_hour=("hour", "max"),
        device_after_hours=("after_hours", "sum"),
        device_weekend=("weekend", "max"),
        device_hour_std=("hour", "std")
    )

    daily["device_hour_std"] = (
        daily["device_hour_std"].fillna(0)
    )

    daily["device_after_hours_ratio"] = (
        daily["device_after_hours"]
        / daily["device_total"]
    )

    daily["connect_disconnect_ratio"] = (
        daily["connect_count"]
        / (daily["disconnect_count"] + 1)
    )

    return daily

## Creating file features

In [ ]:
def build_file_features(dataframe):
    dataframe = prepare_events(dataframe)

    dataframe["filename"] = (
        dataframe["filename"]
        .astype(str)
        .str.strip()
    )

    daily = dataframe.groupby(
        ["user", "day"],
        as_index=False
    ).agg(
        file_total=("filename", "size"),
        unique_files=("filename", "nunique"),
        file_unique_pcs=("pc", "nunique"),
        file_first_hour=("hour", "min"),
        file_last_hour=("hour", "max"),
        file_after_hours=("after_hours", "sum"),
        file_weekend=("weekend", "max"),
        file_hour_std=("hour", "std")
    )

    daily["file_hour_std"] = (
        daily["file_hour_std"].fillna(0)
    )

    daily["file_after_hours_ratio"] = (
        daily["file_after_hours"]
        / daily["file_total"]
    )

    return daily

## Loading and aggregrating the three datasets

In [ ]:
logon_raw = load_columns(
    LOGON_PATH,
    ["date", "user", "pc", "activity"]
)

logon_features = build_logon_features(logon_raw)
del logon_raw

device_raw = load_columns(
    DEVICE_PATH,
    ["date", "user", "pc", "activity"]
)

device_features = build_device_features(device_raw)
del device_raw

file_raw = load_columns(
    FILE_PATH,
    ["date", "user", "pc", "filename"]
)

file_features = build_file_features(file_raw)
del file_raw

print("Logon profiles:", len(logon_features))
print("Device profiles:", len(device_features))
print("File profiles:", len(file_features))

Logon profiles: 1394010
Device profiles: 198993
File profiles: 308647


## Define the model features

These lists were missing from the attached notebook. They explicitly identify which engineered columns enter each autoencoder.


In [ ]:
LOGON_FEATURES = [
    "logon_total",
    "logon_count",
    "logoff_count",
    "logon_unique_pcs",
    "logon_first_hour",
    "logon_last_hour",
    "logon_after_hours",
    "logon_weekend",
    "logon_hour_std",
    "logon_after_hours_ratio",
    "logon_logoff_ratio"
]

DEVICE_FEATURES = [
    "device_total",
    "connect_count",
    "disconnect_count",
    "device_unique_pcs",
    "device_first_hour",
    "device_last_hour",
    "device_after_hours",
    "device_weekend",
    "device_hour_std",
    "device_after_hours_ratio",
    "connect_disconnect_ratio"
]

FILE_FEATURES = [
    "file_total",
    "unique_files",
    "file_unique_pcs",
    "file_first_hour",
    "file_last_hour",
    "file_after_hours",
    "file_weekend",
    "file_hour_std",
    "file_after_hours_ratio"
]

print("Logon model features:", len(LOGON_FEATURES))
print("Device model features:", len(DEVICE_FEATURES))
print("File model features:", len(FILE_FEATURES))


Logon model features: 11
Device model features: 11
File model features: 9


## Defining model features and chronological dates

In [ ]:
logon_features["day"] = pd.to_datetime(logon_features["day"])
device_features["day"] = pd.to_datetime(device_features["day"])
file_features["day"] = pd.to_datetime(file_features["day"])


print(
    "Logon date range:",
    logon_features["day"].min(),
    "to",
    logon_features["day"].max()
)

print(
    "Device date range:",
    device_features["day"].min(),
    "to",
    device_features["day"].max()
)

print(
    "File date range:",
    file_features["day"].min(),
    "to",
    file_features["day"].max()
)


COMMON_START_DAY = max(
    logon_features["day"].min(),
    device_features["day"].min(),
    file_features["day"].min()
)


COMMON_END_DAY = min(
    logon_features["day"].max(),
    device_features["day"].max(),
    file_features["day"].max()
)


if COMMON_START_DAY >= COMMON_END_DAY:
    raise ValueError(
        "The logon, device and file datasets do not have "
        "an overlapping date range. Check that all files "
        "come from the same CERT release."
    )


print(
    "\nCommon date range:",
    COMMON_START_DAY,
    "to",
    COMMON_END_DAY
)


logon_features = logon_features[
    (logon_features["day"] >= COMMON_START_DAY)
    & (logon_features["day"] <= COMMON_END_DAY)
].copy()


device_features = device_features[
    (device_features["day"] >= COMMON_START_DAY)
    & (device_features["day"] <= COMMON_END_DAY)
].copy()


file_features = file_features[
    (file_features["day"] >= COMMON_START_DAY)
    & (file_features["day"] <= COMMON_END_DAY)
].copy()


common_days = pd.date_range(
    start=COMMON_START_DAY,
    end=COMMON_END_DAY,
    freq="D"
)


if len(common_days) < 20:
    raise ValueError(
        "There are too few shared dates for a reliable "
        "training-validation-test split."
    )


train_end_position = int(
    len(common_days) * TRAIN_RATIO
)


validation_end_position = int(
    len(common_days)
    * (TRAIN_RATIO + VALIDATION_RATIO)
)


train_end_position = max(
    1,
    min(train_end_position, len(common_days) - 2)
)

validation_end_position = max(
    train_end_position + 1,
    min(validation_end_position, len(common_days) - 1)
)


TRAIN_END_DAY = common_days[train_end_position]


VALIDATION_END_DAY = common_days[
    validation_end_position
]


print("\nTraining period:")
print(COMMON_START_DAY, "to", TRAIN_END_DAY - pd.Timedelta(days=1))

print("\nValidation period:")
print(
    TRAIN_END_DAY,
    "to",
    VALIDATION_END_DAY - pd.Timedelta(days=1)
)

print("\nTest period:")
print(VALIDATION_END_DAY, "to", COMMON_END_DAY)

Logon date range: 2010-01-02 00:00:00 to 2011-06-01 00:00:00
Device date range: 2010-01-02 00:00:00 to 2011-05-31 00:00:00
File date range: 2010-01-02 00:00:00 to 2011-05-31 00:00:00

Common date range: 2010-01-02 00:00:00 to 2011-05-31 00:00:00

Training period:
2010-01-02 00:00:00 to 2010-12-27 00:00:00

Validation period:
2010-12-28 00:00:00 to 2011-03-14 00:00:00

Test period:
2011-03-15 00:00:00 to 2011-05-31 00:00:00


## split check

In [ ]:
def check_split_sizes(dataframe, dataset_name):
    train_rows = dataframe[
        dataframe["day"] < TRAIN_END_DAY
    ]

    validation_rows = dataframe[
        (dataframe["day"] >= TRAIN_END_DAY)
        & (dataframe["day"] < VALIDATION_END_DAY)
    ]

    test_rows = dataframe[
        dataframe["day"] >= VALIDATION_END_DAY
    ]

    print(f"\n{dataset_name}")
    print("Training rows:", len(train_rows))
    print("Validation rows:", len(validation_rows))
    print("Test rows:", len(test_rows))

    if train_rows.empty:
        raise ValueError(
            f"{dataset_name} training split is empty."
        )

    if validation_rows.empty:
        raise ValueError(
            f"{dataset_name} validation split is empty."
        )

    if test_rows.empty:
        raise ValueError(
            f"{dataset_name} test split is empty."
        )


check_split_sizes(
    logon_features,
    "Logon dataset"
)

check_split_sizes(
    device_features,
    "Device dataset"
)

check_split_sizes(
    file_features,
    "File dataset"
)


Logon dataset
Training rows: 987021
Validation rows: 205061
Test rows: 201915

Device dataset
Training rows: 141472
Validation rows: 29017
Test rows: 28504

File dataset
Training rows: 219713
Validation rows: 44257
Test rows: 44677


## Building the autoencoder

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.encoder = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, input_size)
        )

    def forward(self, values):
        compressed = self.encoder(values)
        reconstructed = self.decoder(compressed)

        return reconstructed

## Calculating the reconstruction error

In [ ]:
def reconstruction_errors(model, values):
    errors = []

    model.eval()

    with torch.no_grad():
        for start in range(0, len(values), BATCH_SIZE):
            batch = values[
                start:start + BATCH_SIZE
            ].to(DEVICE)

            reconstructed = model(batch)

            batch_errors = (
                (batch - reconstructed) ** 2
            ).mean(dim=1)

            errors.extend(
                batch_errors.cpu().numpy()
            )

    return np.array(errors)

## Converting raw scores into comparable percentiles

In [ ]:
def percentile_scores(reference_errors, new_errors):
    sorted_reference = np.sort(reference_errors)

    scores = np.searchsorted(
        sorted_reference,
        new_errors,
        side="right"
    ) / len(sorted_reference)

    return scores

## Training one autoencoder correctly

In [ ]:
def train_stream(dataframe, feature_columns, stream_name):
    dataframe = dataframe.sort_values(
        ["day", "user"]
    ).reset_index(drop=True)

    train_data = dataframe[
        dataframe["day"] < TRAIN_END_DAY
    ].copy()

    validation_data = dataframe[
        (dataframe["day"] >= TRAIN_END_DAY)
        & (dataframe["day"] < VALIDATION_END_DAY)
    ].copy()

    test_data = dataframe[
        dataframe["day"] >= VALIDATION_END_DAY
    ].copy()

    print(f"\nTraining {stream_name} autoencoder")
    print("Training rows:", len(train_data))
    print("Validation rows:", len(validation_data))
    print("Test rows:", len(test_data))

    if train_data.empty:
        raise ValueError(
            f"{stream_name} has no training rows. "
            "Check its date range."
        )

    if validation_data.empty:
        raise ValueError(
            f"{stream_name} has no validation rows. "
            "Check TRAIN_END_DAY and VALIDATION_END_DAY."
        )

    if test_data.empty:
        raise ValueError(
            f"{stream_name} has no test rows. "
            "Check the dataset's final date."
        )

    scaler = StandardScaler()

    train_scaled = scaler.fit_transform(
        train_data[feature_columns]
    )

    validation_scaled = scaler.transform(
        validation_data[feature_columns]
    )

    test_scaled = scaler.transform(
        test_data[feature_columns]
    )

    train_values = torch.tensor(
        train_scaled,
        dtype=torch.float32
    )

    validation_values = torch.tensor(
        validation_scaled,
        dtype=torch.float32
    )

    test_values = torch.tensor(
        test_scaled,
        dtype=torch.float32
    )

    torch.manual_seed(SEED)

    model = Autoencoder(
        len(feature_columns)
    ).to(DEVICE)

    loss_function = nn.MSELoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    best_validation_loss = np.inf
    best_weights = None
    waiting_epochs = 0

    for epoch in range(EPOCHS):
        model.train()

        row_order = torch.randperm(
            len(train_values)
        )

        total_training_loss = 0

        for start in range(
            0,
            len(train_values),
            BATCH_SIZE
        ):
            row_numbers = row_order[
                start:start + BATCH_SIZE
            ]

            batch = train_values[
                row_numbers
            ].to(DEVICE)

            optimizer.zero_grad()

            reconstructed = model(batch)

            loss = loss_function(
                reconstructed,
                batch
            )

            loss.backward()
            optimizer.step()

            total_training_loss += (
                loss.item() * len(batch)
            )

        average_training_loss = (
            total_training_loss
            / len(train_values)
        )

        current_validation_errors = (
            reconstruction_errors(
                model,
                validation_values
            )
        )

        current_validation_loss = (
            current_validation_errors.mean()
        )

        if current_validation_loss < best_validation_loss:
            best_validation_loss = current_validation_loss

            best_weights = {
                name: value.detach().cpu().clone()
                for name, value
                in model.state_dict().items()
            }

            waiting_epochs = 0

        else:
            waiting_epochs += 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                stream_name,
                "epoch",
                epoch + 1,
                "training loss",
                round(average_training_loss, 6),
                "validation loss",
                round(current_validation_loss, 6)
            )

        if waiting_epochs >= PATIENCE:
            print(stream_name, "stopped at epoch", epoch + 1)
            break

    model.load_state_dict(best_weights)

    validation_errors = reconstruction_errors(
        model,
        validation_values
    )

    test_errors = reconstruction_errors(
        model,
        test_values
    )

    threshold = np.quantile(
        validation_errors,
        STREAM_THRESHOLD_PERCENTILE
    )

    validation_results = validation_data[
        ["user", "day"]
    ].reset_index(drop=True)

    validation_results[
        f"{stream_name}_score"
    ] = validation_errors

    validation_results[
        f"{stream_name}_percentile"
    ] = percentile_scores(
        validation_errors,
        validation_errors
    )

    validation_results[
        f"{stream_name}_flag"
    ] = (
        validation_errors > threshold
    ).astype(int)

    test_results = test_data[
        ["user", "day"]
    ].reset_index(drop=True)

    test_results[
        f"{stream_name}_score"
    ] = test_errors

    test_results[
        f"{stream_name}_percentile"
    ] = percentile_scores(
        validation_errors,
        test_errors
    )

    test_results[
        f"{stream_name}_flag"
    ] = (
        test_errors > threshold
    ).astype(int)

    return {
        "model": model,
        "scaler": scaler,
        "threshold": float(threshold),
        "validation": validation_results,
        "test": test_results
    }

## Training the three independent models

In [ ]:
logon_output = train_stream(
    logon_features,
    LOGON_FEATURES,
    "logon"
)

device_output = train_stream(
    device_features,
    DEVICE_FEATURES,
    "device"
)

file_output = train_stream(
    file_features,
    FILE_FEATURES,
    "file"
)


Training logon autoencoder
Training rows: 987021
Validation rows: 205061
Test rows: 201915
logon epoch 1 training loss 0.239513 validation loss 0.029241
logon epoch 10 training loss 0.001384 validation loss 0.001255
logon epoch 20 training loss 0.000799 validation loss 0.000763
logon epoch 30 training loss 0.000642 validation loss 0.000531
logon epoch 40 training loss 0.000471 validation loss 0.000734
logon epoch 50 training loss 0.000365 validation loss 0.000367

Training device autoencoder
Training rows: 141472
Validation rows: 29017
Test rows: 28504
device epoch 1 training loss 0.89091 validation loss 0.653598
device epoch 10 training loss 0.008944 validation loss 0.008824
device epoch 20 training loss 0.004567 validation loss 0.004397
device epoch 30 training loss 0.002422 validation loss 0.003162
device epoch 40 training loss 0.001991 validation loss 0.002145
device epoch 50 training loss 0.00166 validation loss 0.00177

Training file autoencoder
Training rows: 219713
Validation 

## Merging hte model scores and Creating the ensemble

In [ ]:
validation_ensemble = (
    logon_output["validation"]
    .merge(
        device_output["validation"],
        on=["user", "day"],
        how="outer"
    )
    .merge(
        file_output["validation"],
        on=["user", "day"],
        how="outer"
    )
)

test_ensemble = (
    logon_output["test"]
    .merge(
        device_output["test"],
        on=["user", "day"],
        how="outer"
    )
    .merge(
        file_output["test"],
        on=["user", "day"],
        how="outer"
    )
)

percentile_columns = [
    "logon_percentile",
    "device_percentile",
    "file_percentile"
]

flag_columns = [
    "logon_flag",
    "device_flag",
    "file_flag"
]

validation_ensemble["available_streams"] = (
    validation_ensemble[percentile_columns]
    .notna()
    .sum(axis=1)
)

test_ensemble["available_streams"] = (
    test_ensemble[percentile_columns]
    .notna()
    .sum(axis=1)
)

validation_ensemble["ensemble_score"] = (
    validation_ensemble[percentile_columns]
    .mean(axis=1, skipna=True)
)

test_ensemble["ensemble_score"] = (
    test_ensemble[percentile_columns]
    .mean(axis=1, skipna=True)
)

ensemble_threshold = np.quantile(
    validation_ensemble["ensemble_score"].dropna(),
    ENSEMBLE_THRESHOLD_PERCENTILE
)

# Keep missing stream flags as missing rather than changing them to zero.
for column in flag_columns:
    test_ensemble[column] = (
        test_ensemble[column]
        .astype("Int64")
    )

test_ensemble["ensemble_flag"] = (
    test_ensemble["ensemble_score"]
    > ensemble_threshold
).astype(int)

test_ensemble["flagged_streams"] = (
    test_ensemble[flag_columns]
    .eq(1)
    .sum(axis=1)
)

print("Ensemble threshold:", ensemble_threshold)
print(
    "Detected test anomalies:",
    test_ensemble["ensemble_flag"].sum()
)
print(
    "Available stream counts:"
)
print(
    test_ensemble["available_streams"]
    .value_counts()
    .sort_index()
)


Ensemble threshold: 0.9293635547462217
Detected test anomalies: 9956
Available stream counts:
available_streams
1    153329
2     23991
3     24595
Name: count, dtype: int64


## Creating the technical anomaly categories

In [ ]:
def technical_category(row):
    if row["ensemble_flag"] == 0:
        return "Normal"

    available_streams = []

    for stream in ["logon", "device", "file"]:
        if not pd.isna(row[f"{stream}_percentile"]):
            available_streams.append(stream)

    if len(available_streams) == 0:
        return "Unclassified Technical Anomaly"

    if len(available_streams) == 1:
        return (
            available_streams[0].capitalize()
            + "-Only Anomaly"
        )

    if row["flagged_streams"] >= 2:
        return "Multi-Stream Anomaly"

    dominant_stream = max(
        available_streams,
        key=lambda stream:
        row[f"{stream}_percentile"]
    )

    return (
        dominant_stream.capitalize()
        + "-Dominant Anomaly"
    )


test_ensemble["technical_category"] = (
    test_ensemble.apply(
        technical_category,
        axis=1
    )
)


## Loading and preparing the psychometric file

In [ ]:
psychometric = pd.read_csv(
    PSYCHOMETRIC_PATH,
    low_memory=False
)

psychometric.columns = [
    column.strip().lower()
    for column in psychometric.columns
]

if "user_id" in psychometric.columns:
    psychometric = psychometric.rename(
        columns={"user_id": "user"}
    )

elif "userid" in psychometric.columns:
    psychometric = psychometric.rename(
        columns={"userid": "user"}
    )

elif "user" not in psychometric.columns:
    raise ValueError(
        "No user ID column was found in psychometric.csv."
    )

required_ocean_columns = [
    "o",
    "c",
    "e",
    "a",
    "n"
]

missing_ocean_columns = [
    column
    for column in required_ocean_columns
    if column not in psychometric.columns
]

if missing_ocean_columns:
    raise ValueError(
        f"Missing OCEAN columns: {missing_ocean_columns}"
    )

psychometric = psychometric.rename(
    columns={
        "o": "openness",
        "c": "conscientiousness",
        "e": "extraversion",
        "a": "agreeableness",
        "n": "neuroticism"
    }
)

psychometric["user"] = (
    psychometric["user"]
    .astype(str)
    .str.strip()
    .str.upper()
)

OCEAN_TRAITS = [
    "openness",
    "conscientiousness",
    "extraversion",
    "agreeableness",
    "neuroticism"
]

for trait in OCEAN_TRAITS:
    psychometric[trait] = pd.to_numeric(
        psychometric[trait],
        errors="coerce"
    )

psychometric = (
    psychometric
    .dropna(subset=OCEAN_TRAITS)
    .drop_duplicates(subset=["user"])
)

## Calculating the OCEAN bands using training users

In [ ]:
training_users = set(
    pd.concat(
        [
            logon_features[
                logon_features["day"] < TRAIN_END_DAY
            ]["user"],

            device_features[
                device_features["day"] < TRAIN_END_DAY
            ]["user"],

            file_features[
                file_features["day"] < TRAIN_END_DAY
            ]["user"]
        ],
        ignore_index=True
    ).unique()
)

psychometric_reference = psychometric[
    psychometric["user"].isin(training_users)
].copy()

if psychometric_reference.empty:
    raise ValueError(
        "No psychometric users matched the activity datasets."
    )

low_cutoffs = psychometric_reference[
    OCEAN_TRAITS
].quantile(OCEAN_LOW_PERCENTILE)

high_cutoffs = psychometric_reference[
    OCEAN_TRAITS
].quantile(OCEAN_HIGH_PERCENTILE)

final_results = test_ensemble.merge(
    psychometric,
    on="user",
    how="left"
)

print(
    "Psychometric match rate:",
    round(
        final_results["openness"].notna().mean() * 100,
        2
    ),
    "%"
)

Psychometric match rate: 100.0 %


## assigning the psychometric categories

In [ ]:
def trait_band(value, trait):
    if pd.isna(value):
        return "Unavailable"

    if value <= low_cutoffs[trait]:
        return "Low"

    if value >= high_cutoffs[trait]:
        return "High"

    return "Typical"


for trait in OCEAN_TRAITS:
    final_results[
        f"{trait}_band"
    ] = final_results[trait].apply(
        lambda value, current_trait=trait:
        trait_band(value, current_trait)
    )


trait_names = {
    "openness": "Openness",
    "conscientiousness": "Conscientiousness",
    "extraversion": "Extraversion",
    "agreeableness": "Agreeableness",
    "neuroticism": "Neuroticism"
}


def ocean_context(row):
    if row["ensemble_flag"] == 0:
        return "Not Applied to Normal Record"

    if pd.isna(row["openness"]):
        return "Psychometric Data Unavailable"

    labels = []

    for trait in OCEAN_TRAITS:
        band = row[f"{trait}_band"]

        if band == "Low" or band == "High":
            labels.append(
                band + " " + trait_names[trait] + " Profile"
            )

    if not labels:
        return "No Extreme OCEAN Trait"

    return " | ".join(labels)


final_results["ocean_context"] = (
    final_results.apply(
        ocean_context,
        axis=1
    )
)

## Creating the final combined category

In [ ]:
def final_category(row):
    if row["ensemble_flag"] == 0:
        return "Normal"

    return (
        row["technical_category"]
        + " | "
        + row["ocean_context"]
    )


final_results["final_category"] = (
    final_results.apply(
        final_category,
        axis=1
    )
)

## Arranging and Saving the results

In [ ]:
output_columns = [
    "user",
    "day",

    "logon_score",
    "logon_percentile",
    "logon_flag",

    "device_score",
    "device_percentile",
    "device_flag",

    "file_score",
    "file_percentile",
    "file_flag",

    "ensemble_score",
    "ensemble_flag",
    "available_streams",
    "flagged_streams",
    "technical_category",

    "openness",
    "openness_band",

    "conscientiousness",
    "conscientiousness_band",

    "extraversion",
    "extraversion_band",

    "agreeableness",
    "agreeableness_band",

    "neuroticism",
    "neuroticism_band",

    "ocean_context",
    "final_category"
]

final_output = final_results[
    output_columns
].sort_values(
    "ensemble_score",
    ascending=False
).reset_index(drop=True)

RESULT_PATH = os.path.join(
    OUTPUT_FOLDER,
    "ensemble_anomaly_results.csv"
)

THRESHOLD_PATH = os.path.join(
    OUTPUT_FOLDER,
    "model_thresholds.csv"
)

final_output.to_csv(
    RESULT_PATH,
    index=False
)

threshold_table = pd.DataFrame(
    {
        "model": [
            "logon",
            "device",
            "file",
            "ensemble"
        ],
        "threshold": [
            logon_output["threshold"],
            device_output["threshold"],
            file_output["threshold"],
            ensemble_threshold
        ]
    }
)

threshold_table.to_csv(
    THRESHOLD_PATH,
    index=False
)

torch.save(
    logon_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "logon_autoencoder.pth"
    )
)

torch.save(
    device_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "device_autoencoder.pth"
    )
)

torch.save(
    file_output["model"].state_dict(),
    os.path.join(
        OUTPUT_FOLDER,
        "file_autoencoder.pth"
    )
)

pd.to_pickle(
    logon_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "logon_scaler.pkl"
    )
)

pd.to_pickle(
    device_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "device_scaler.pkl"
    )
)

pd.to_pickle(
    file_output["scaler"],
    os.path.join(
        OUTPUT_FOLDER,
        "file_scaler.pkl"
    )
)

print("Results saved to:", RESULT_PATH)
print("Thresholds saved to:", THRESHOLD_PATH)

final_output.head(20)


Results saved to: /content/drive/MyDrive/r6.2/CERT_model_outputs/ensemble_anomaly_results.csv
Thresholds saved to: /content/drive/MyDrive/r6.2/CERT_model_outputs/model_thresholds.csv


,user,day,logon_score,logon_percentile,logon_flag,device_score,device_percentile,device_flag,file_score,file_percentile,...,conscientiousness,conscientiousness_band,extraversion,extraversion_band,agreeableness,agreeableness_band,neuroticism,neuroticism_band,ocean_context,final_category
0,TAM3048,2011-04-17,0.207287,0.999941,1,NaN,NaN,<NA>,NaN,NaN,...,25,Typical,39,Typical,43,Typical,29,Typical,No Extreme OCEAN Trait,Logon-Only Anomaly | No Extreme OCEAN Trait
1,ALC1060,2011-04-18,0.166968,0.999893,1,NaN,NaN,<NA>,NaN,NaN,...,49,High,38,Typical,35,Typical,28,Typical,High Conscientiousness Profile,Logon-Only Anomaly | High Conscientiousness Pr...
2,DNS1758,2011-04-05,0.160712,0.999873,1,NaN,NaN,<NA>,NaN,NaN,...,37,Typical,35,Typical,15,Typical,30,Typical,No Extreme OCEAN Trait,Logon-Only Anomaly | No Extreme OCEAN Trait
3,BEH0103,2011-05-21,0.138358,0.999839,1,NaN,NaN,<NA>,NaN,NaN,...,43,Typical,43,Typical,19,Typical,32,Typical,No Extreme OCEAN Trait,Logon-Only Anomaly | No Extreme OCEAN Trait
4,TAM3048,2011-03-27,0.137857,0.999839,1,NaN,NaN,<NA>,NaN,NaN,...,25,Typical,39,Typical,43,Typical,29,Typical,No Extreme OCEAN Trait,Logon-Only Anomaly | No Extreme OCEAN Trait
5,ZRM0694,2011-05-01,0.123545,0.999790,1,NaN,NaN,<NA>,NaN,NaN,...,14,Low,18,Typical,45,Typical,34,Typical,Low Conscientiousness Profile,Logon-Only Anomaly | Low Conscientiousness Pro...
6,RCF0044,2011-05-22,0.123932,0.999790,1,NaN,NaN,<NA>,NaN,NaN,...,35,Typical,38,Typical,21,Typical,28,Typical,No Extreme OCEAN Trait,Logon-Only Anomaly | No Extreme OCEAN Trait
7,EUC1051,2011-05-13,0.109365,0.999756,1,NaN,NaN,<NA>,NaN,NaN,...,18,Typical,18,Typical,15,Typical,30,Typical,High Openness Profile,Logon-Only Anomaly | High Openness Profile
8,YKM3542,2011-04-23,0.098055,0.999698,1,NaN,NaN,<NA>,NaN,NaN,...,44,Typical,22,Typical,42,Typical,27,Typical,No Extreme OCEAN Trait,Logon-Only Anomaly | No Extreme OCEAN Trait
9,MTS0465,2011-05-01,0.094463,0.999673,1,NaN,NaN,<NA>,NaN,NaN,...,39,Typical,44,Typical,41,Typical,33,Typical,High Openness Profile,Logon-Only Anomaly | High Openness Profile


# Improvements after the original model

The original model ends above. The cells below evaluate the test results against the verified `ground_truth.csv`.


In [ ]:
GROUND_TRUTH_PATH = os.path.join(
    DATA_FOLDER,
    "ground_truth.csv"
)

if not os.path.exists(GROUND_TRUTH_PATH):
    print(
        "Evaluation skipped: ground_truth.csv was not found in",
        DATA_FOLDER
    )
    print(
        "Create it from the official answer material, then run "
        "the evaluation cells again."
    )
else:
    print("Ground truth found:", GROUND_TRUTH_PATH)


Evaluation skipped: ground_truth.csv was not found in /content/drive/MyDrive/r6.2
Create it from the official answer material, then run the evaluation cells again.


## Load and validate ground truth


In [ ]:
ground_truth = pd.read_csv(GROUND_TRUTH_PATH)

required_ground_truth_columns = [
    "user",
    "day",
    "scenario_id"
]

missing_columns = [
    column
    for column in required_ground_truth_columns
    if column not in ground_truth.columns
]

if missing_columns:
    raise ValueError(
        "ground_truth.csv is missing: "
        + str(missing_columns)
    )

ground_truth["user"] = (
    ground_truth["user"]
    .astype(str)
    .str.strip()
    .str.upper()
)

ground_truth["day"] = (
    pd.to_datetime(
        ground_truth["day"],
        errors="coerce"
    )
    .dt.normalize()
)

if ground_truth["day"].isna().any():
    raise ValueError(
        "Some ground-truth dates could not be parsed."
    )

ground_truth = ground_truth.drop_duplicates(
    subset=["user", "day", "scenario_id"]
).reset_index(drop=True)

daily_ground_truth = (
    ground_truth
    .groupby(
        ["user", "day"],
        as_index=False
    )
    .agg(
        ground_truth_label=("scenario_id", "size"),
        scenario_id=(
            "scenario_id",
            lambda values:
            " | ".join(sorted(set(values.astype(str))))
        )
    )
)

daily_ground_truth["ground_truth_label"] = 1

print("Malicious user-days:", len(daily_ground_truth))
print(
    "Malicious users:",
    daily_ground_truth["user"].nunique()
)


## Locate the answers archive

In [ ]:
import glob
import os

ANSWER_ARCHIVE_PATH = os.path.join(
    DATA_FOLDER,
    "answers.tar.bz2"
)

if not os.path.exists(ANSWER_ARCHIVE_PATH):
    answer_candidates = glob.glob(
        DRIVE_ROOT + "/**/answers.tar.bz2",
        recursive=True
    )

    if len(answer_candidates) == 0:
        raise FileNotFoundError(
            "answers.tar.bz2 was not found in My Drive."
        )

    if len(answer_candidates) > 1:
        print("Multiple answer archives were found:")

        for path in answer_candidates:
            print(path)

        raise ValueError(
            "Set ANSWER_ARCHIVE_PATH manually using "
            "the correct path printed above."
        )

    ANSWER_ARCHIVE_PATH = answer_candidates[0]

print(
    "Answers archive:",
    ANSWER_ARCHIVE_PATH
)

Answers archive: /content/drive/MyDrive/r6.2/answers.tar.bz2


## Creating ground_truth.csv

In [ ]:
import csv
import io
import tarfile

CERT_RELEASE = "6.2"

ground_truth_records = []

with tarfile.open(
    ANSWER_ARCHIVE_PATH,
    mode="r:bz2"
) as archive:

    insiders_file = archive.extractfile(
        "answers/insiders.csv"
    )

    if insiders_file is None:
        raise FileNotFoundError(
            "answers/insiders.csv was not found "
            "inside the archive."
        )

    insiders = pd.read_csv(
        insiders_file,
        dtype=str
    )

    release_incidents = insiders[
        insiders["dataset"] == CERT_RELEASE
    ].copy()

    if release_incidents.empty:
        raise ValueError(
            "No answer records were found for CERT "
            + CERT_RELEASE
        )

    print(
        "Incidents found:",
        len(release_incidents)
    )

    for _, incident in release_incidents.iterrows():

        insider_user = (
            incident["user"]
            .strip()
            .upper()
        )

        details_filename = incident["details"]

        details_path = (
            "answers/"
            + details_filename
        )

        details_file = archive.extractfile(
            details_path
        )

        if details_file is None:
            raise FileNotFoundError(
                "Missing answer file: "
                + details_path
            )

        text_file = io.TextIOWrapper(
            details_file,
            encoding="utf-8",
            errors="replace",
            newline=""
        )

        rows = csv.reader(text_file)

        for row in rows:

            if len(row) < 4:
                continue

            event_date = pd.to_datetime(
                row[2],
                format=DATE_FORMAT,
                errors="coerce"
            )

            event_user = (
                row[3]
                .strip()
                .upper()
            )

            if pd.isna(event_date):
                continue

            if event_user != insider_user:
                continue

            ground_truth_records.append(
                {
                    "user": insider_user,
                    "day": event_date.normalize(),
                    "scenario_id": (
                        "r"
                        + CERT_RELEASE
                        + "-scenario-"
                        + str(incident["scenario"])
                    )
                }
            )

Incidents found: 5


## Removing duplicate user-days

In [ ]:
ground_truth = pd.DataFrame(
    ground_truth_records
)

if ground_truth.empty:
    raise ValueError(
        "No malicious user-days were extracted."
    )

ground_truth = (
    ground_truth
    .drop_duplicates(
        subset=[
            "user",
            "day",
            "scenario_id"
        ]
    )
    .sort_values(
        [
            "day",
            "user",
            "scenario_id"
        ]
    )
    .reset_index(drop=True)
)

GROUND_TRUTH_PATH = os.path.join(
    DATA_FOLDER,
    "ground_truth.csv"
)

ground_truth.to_csv(
    GROUND_TRUTH_PATH,
    index=False
)

print(
    "Ground truth saved to:",
    GROUND_TRUTH_PATH
)

print(
    "Malicious users:",
    ground_truth["user"].nunique()
)

print(
    "Malicious user-days:",
    len(ground_truth)
)

ground_truth

Ground truth saved to: /content/drive/MyDrive/r6.2/ground_truth.csv
Malicious users: 5
Malicious user-days: 44


,user,day,scenario_id
0,PLJ1771,2010-08-12,r6.2-scenario-3
1,PLJ1771,2010-08-13,r6.2-scenario-3
2,ACM2278,2010-08-18,r6.2-scenario-1
3,ACM2278,2010-08-19,r6.2-scenario-1
4,ACM2278,2010-08-24,r6.2-scenario-1
5,MBG3183,2010-10-12,r6.2-scenario-5
6,CMP2946,2011-02-02,r6.2-scenario-2
7,CMP2946,2011-02-03,r6.2-scenario-2
8,CMP2946,2011-02-04,r6.2-scenario-2
9,CMP2946,2011-02-07,r6.2-scenario-2


## Creating daily ground truth

In [ ]:
daily_ground_truth = (
    ground_truth
    .groupby(
        ["user", "day"],
        as_index=False
    )
    .agg(
        scenario_id=(
            "scenario_id",
            lambda values:
            " | ".join(
                sorted(
                    set(
                        values.astype(str)
                    )
                )
            )
        )
    )
)

daily_ground_truth[
    "ground_truth_label"
] = 1

print(
    "Daily ground-truth rows:",
    len(daily_ground_truth)
)

print(
    "Malicious users:",
    daily_ground_truth["user"].nunique()
)

daily_ground_truth.head()

Daily ground-truth rows: 44
Malicious users: 5


,user,day,scenario_id,ground_truth_label
0,ACM2278,2010-08-18,r6.2-scenario-1,1
1,ACM2278,2010-08-19,r6.2-scenario-1,1
2,ACM2278,2010-08-24,r6.2-scenario-1,1
3,CDE1846,2011-02-21,r6.2-scenario-4,1
4,CDE1846,2011-03-17,r6.2-scenario-4,1


## Merge labels and confirm test coverage


In [ ]:
evaluation_results = final_output.merge(
    daily_ground_truth,
    on=["user", "day"],
    how="left"
)

evaluation_results["ground_truth_label"] = (
    evaluation_results["ground_truth_label"]
    .fillna(0)
    .astype(int)
)

evaluation_results["scenario_id"] = (
    evaluation_results["scenario_id"]
    .fillna("Background")
)

malicious_test_rows = int(
    evaluation_results["ground_truth_label"].sum()
)

print("Test rows:", len(evaluation_results))
print("Malicious test user-days:", malicious_test_rows)

if malicious_test_rows == 0:
    raise ValueError(
        "The test period contains no verified malicious user-days. "
        "Performance cannot be evaluated with this split."
    )


Test rows: 201915
Malicious test user-days: 12


## Calculate performance metrics


In [ ]:
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score
)

y_true = evaluation_results["ground_truth_label"]
y_pred = evaluation_results["ensemble_flag"]
y_score = evaluation_results["ensemble_score"]

tn, fp, fn, tp = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1]
).ravel()

precision = precision_score(
    y_true,
    y_pred,
    zero_division=0
)

recall = recall_score(
    y_true,
    y_pred,
    zero_division=0
)

f1 = f1_score(
    y_true,
    y_pred,
    zero_division=0
)

balanced_accuracy = balanced_accuracy_score(
    y_true,
    y_pred
)

average_precision = average_precision_score(
    y_true,
    y_score
)

false_positive_rate = fp / (fp + tn)

alerts_per_day = (
    evaluation_results
    .groupby("day")["ensemble_flag"]
    .sum()
    .mean()
)

metrics_table = pd.DataFrame(
    {
        "metric": [
            "True positives",
            "False positives",
            "False negatives",
            "True negatives",
            "Precision",
            "Recall",
            "F1-score",
            "Balanced accuracy",
            "Average precision",
            "False-positive rate",
            "Average alerts per day"
        ],
        "value": [
            tp,
            fp,
            fn,
            tn,
            precision,
            recall,
            f1,
            balanced_accuracy,
            average_precision,
            false_positive_rate,
            alerts_per_day
        ]
    }
)

metrics_table


,metric,value
0,True positives,2.000000
1,False positives,9954.000000
2,False negatives,10.000000
3,True negatives,191949.000000
4,Precision,0.000201
5,Recall,0.166667
6,F1-score,0.000401
7,Balanced accuracy,0.558683
8,Average precision,0.000306
9,False-positive rate,0.049301


## Compare individual streams with the ensemble


In [ ]:
def evaluate_scores(
    dataframe,
    score_column,
    flag_column,
    model_name
):
    available = dataframe.dropna(
        subset=[score_column, flag_column]
    ).copy()

    true_values = available["ground_truth_label"]
    predictions = available[flag_column].astype(int)
    scores = available[score_column]

    if true_values.nunique() == 2:
        model_ap = average_precision_score(
            true_values,
            scores
        )
    else:
        model_ap = np.nan

    return {
        "model": model_name,
        "rows_evaluated": len(available),
        "malicious_rows": int(true_values.sum()),
        "precision": precision_score(
            true_values,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            true_values,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            true_values,
            predictions,
            zero_division=0
        ),
        "average_precision": model_ap
    }


model_comparison = pd.DataFrame(
    [
        evaluate_scores(
            evaluation_results,
            "logon_percentile",
            "logon_flag",
            "Logon autoencoder"
        ),
        evaluate_scores(
            evaluation_results,
            "device_percentile",
            "device_flag",
            "Device autoencoder"
        ),
        evaluate_scores(
            evaluation_results,
            "file_percentile",
            "file_flag",
            "File autoencoder"
        ),
        evaluate_scores(
            evaluation_results,
            "ensemble_score",
            "ensemble_flag",
            "Ensemble"
        )
    ]
)

model_comparison


,model,rows_evaluated,malicious_rows,precision,recall,f1,average_precision
0,Logon autoencoder,201915,12,0.000199,0.166667,0.000398,0.000264
1,Device autoencoder,28504,4,0.000000,0.000000,0.000000,0.000423
2,File autoencoder,44677,12,0.001779,0.333333,0.003538,0.005345
3,Ensemble,201915,12,0.000201,0.166667,0.000401,0.000306
